In [3]:
import sympy as sp
import numpy as np
import sys

def fixed_point_iteration():
    print("=== CHƯƠNG TRÌNH GIẢI PHƯƠNG TRÌNH BẰNG PHƯƠNG PHÁP LẶP ĐƠN ===")
    
    try:
        # --- 0. Tương tác nhập liệu ---
        raw_phi = input("Nhập hàm lặp phi(x) (ví dụ: 1/sqrt(x+3) hoặc (x+1)**(1/3)): ")
        phi_str = raw_phi.replace('^', '**')
        
        a = float(input("Nhập đầu mút a: "))
        b = float(input("Nhập đầu mút b: "))
        
        if a > b:
            a, b = b, a
            
        x0 = float(input("Nhập xấp xỉ ban đầu x0: "))
        epsilon = float(input("Nhập sai số mục tiêu epsilon (ví dụ: 1e-5): "))
        precision = int(input("Nhập số chữ số thập phân cần làm tròn (ví dụ: 7): "))
        
        print("\nChọn công thức đánh giá sai số:")
        print("  a - Sai số tiên nghiệm (dựa vào xấp xỉ ban đầu)")
        print("  b - Sai số hậu nghiệm (dựa vào 2 xấp xỉ liên tiếp)")
        option = input("Nhập 'a' hoặc 'b': ").strip().lower()

        # --- 1. Xử lý toán học với SymPy ---
        x = sp.Symbol('x')
        phi = sp.sympify(phi_str)
        phi_prime = sp.diff(phi, x)
        
        # Tìm hệ số co q trên [a, b]
        x_vals = np.linspace(a, b, 1000)
        phi_prime_func = sp.lambdify(x, phi_prime, 'numpy')
        q_vals = np.abs(phi_prime_func(x_vals))
        q = float(np.max(q_vals))
        
        if q >= 1:
            print(f"\n[!] CẢNH BÁO CHÍ MẠNG: Hệ số co q = {q:.4f} >= 1.")
            print(f"Phương pháp lặp đơn PHÂN KỲ (không hội tụ) trên khoảng [{a}, {b}]. Vui lòng chọn hàm phi(x) khác hoặc thu hẹp khoảng.")
            return
        
        phi_func = sp.lambdify(x, phi, 'numpy')
        
        # --- 2. Quá trình lặp ---
        xn = x0
        x_history = [xn]
        errors = [0.0]
        
        x1 = float(phi_func(x0))
        n = 1
        
        while True:
            xn_next = float(phi_func(xn))
            x_history.append(xn_next)
            
            if option == 'a':
                # Sai số tiên nghiệm
                err = (q**n / (1 - q)) * abs(x1 - x0)
            else:
                # Sai số hậu nghiệm
                err = (q / (1 - q)) * abs(xn_next - xn)
                
            errors.append(err)
            
            if err <= epsilon or n > 1000:
                break
                
            xn = xn_next
            n += 1

        total_steps = n
        
        # --- 3. TRÌNH BÀY KẾT QUẢ ---
        print("\n" + "="*60)
        print("### 1. Thiết lập hàm lặp và thông số")
        print(f"* **Hàm lặp:** $\\varphi(x) = {sp.latex(phi)}$")
        print(f"* **Đạo hàm:** $\\varphi'(x) = {sp.latex(phi_prime)}$")
        print(f"* **Hệ số co $q$:** $q = \\max_{{x \\in [{a}, {b}]}} |\\varphi'(x)| \\approx {q:.{precision}f} < 1$.")
        print(f"* **Xấp xỉ ban đầu:** $x_0 = {x0:.{precision}f}$.")
        
        print("\n### 2. Xây dựng công thức")
        print("* **Công thức lặp:**")
        print("  $$x_n = \\varphi(x_{n-1})$$")
        
        if option == 'a':
            print("* **Công thức sai số (Tiên nghiệm):**")
            print(f"  $$\\Delta_n = \\frac{{{q:.{precision}f}^n}}{{1 - {q:.{precision}f}}} |x_1 - x_0| \\le {epsilon:.1e}$$")
        else:
            print("* **Công thức sai số (Hậu nghiệm):**")
            print(f"  $$\\Delta_n = \\frac{{{q:.{precision}f}}}{{1 - {q:.{precision}f}}} |x_n - x_{{n-1}}| \\le {epsilon:.1e}$$")

        print("\n### 3. Chi tiết các bước lặp tiêu biểu\n")
        
        # 2 BƯỚC ĐẦU
        print("**Hai bước lặp đầu tiên:**")
        for i in range(1, min(3, total_steps + 1)):
            print(f"* **Bước $n={i}$:**")
            print(f"    $$x_{{{i}}} = \\varphi({x_history[i-1]:.{precision}f}) = {x_history[i]:.{precision}f}$$")
            print(f"    $$\\Delta_{{{i}}} = {errors[i]:.{precision}g}$$")

        # 2 BƯỚC CUỐI
        if total_steps > 2:
            print("\n**Hai bước lặp cuối cùng:**")
            for i in range(max(3, total_steps - 1), total_steps + 1):
                print(f"* **Bước $n={i}$:**")
                print(f"    $$x_{{{i}}} = \\varphi({x_history[i-1]:.{precision}f}) = {x_history[i]:.{precision}f}$$")
                print(f"    $$\\Delta_{{{i}}} = {errors[i]:.{precision}g}$$")

        print("\n### 4. Bảng lặp tổng hợp\n")
        print("| $n$ | $x_n$ | $\\Delta_n$ |")
        print("| :--- | :--- | :--- |")
        
        # In 3 dòng đầu
        for i in range(min(3, total_steps + 1)):
            err_str = f"{errors[i]:.{precision}g}" if i > 0 else "-"
            print(f"| {i} | {x_history[i]:.{precision}f} | {err_str} |")
            
        if total_steps >= 6:
            print("| ... | ... | ... |")
            
        # In 3 dòng cuối
        start_tail = max(3, total_steps - 2)
        if start_tail <= min(3, total_steps + 1) - 1:
            start_tail = min(3, total_steps + 1)
            
        for i in range(start_tail, total_steps + 1):
            print(f"| {i} | {x_history[i]:.{precision}f} | {errors[i]:.{precision}g} |")

        print(f"\n=> **KẾT LUẬN:** Nghiệm xấp xỉ của phương trình là **$x^* \\approx {x_history[-1]:.{precision}f}$** (với sai số $\\Delta_n = {errors[-1]:.{precision}g} \\le {epsilon:.1e}$).")

    except Exception as e:
        print(f"\n[!] CÓ LỖI XẢY RA TRONG QUÁ TRÌNH TÍNH TOÁN:")
        print(f"Chi tiết lỗi: {e}")

if __name__ == "__main__":
    fixed_point_iteration()
    input("\nNhấn Enter để thoát chương trình...")

=== CHƯƠNG TRÌNH GIẢI PHƯƠNG TRÌNH BẰNG PHƯƠNG PHÁP LẶP ĐƠN ===

Chọn công thức đánh giá sai số:
  a - Sai số tiên nghiệm (dựa vào xấp xỉ ban đầu)
  b - Sai số hậu nghiệm (dựa vào 2 xấp xỉ liên tiếp)

### 1. Thiết lập hàm lặp và thông số
* **Hàm lặp:** $\varphi(x) = \frac{1}{\sqrt{x + 3}}$
* **Đạo hàm:** $\varphi'(x) = - \frac{1}{2 \left(x + 3\right)^{\frac{3}{2}}}$
* **Hệ số co $q$:** $q = \max_{x \in [0.0, 1.0]} |\varphi'(x)| \approx 0.0962250 < 1$.
* **Xấp xỉ ban đầu:** $x_0 = 0.3000000$.

### 2. Xây dựng công thức
* **Công thức lặp:**
  $$x_n = \varphi(x_{n-1})$$
* **Công thức sai số (Tiên nghiệm):**
  $$\Delta_n = \frac{0.0962250^n}{1 - 0.0962250} |x_1 - x_0| \le 1.0e-09$$

### 3. Chi tiết các bước lặp tiêu biểu

**Hai bước lặp đầu tiên:**
* **Bước $n=1$:**
    $$x_{1} = \varphi(0.3000000) = 0.5504819$$
    $$\Delta_{1} = 0.02666884$$
* **Bước $n=2$:**
    $$x_{2} = \varphi(0.5504819) = 0.5307089$$
    $$\Delta_{2} = 0.00256621$$

**Hai bước lặp cuối cùng:**
* **Bước $n=8$:**
    